Component ablation A2

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

acoustic_dataset_1 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
acoustic_dataset = acoustic_dataset_1.drop(columns=["Participant_ID"])
print(acoustic_dataset.shape)
print(acoustic_dataset.head())

(193, 70)
   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0   118.719333    141.312833    106.029833        0.010085          0.000087   
1   113.014000    154.201333    105.652667        0.008043          0.000072   
2   223.657833    239.034500    161.921500        0.002585          0.000010   
3   141.336667    165.070500     66.233833        0.009008          0.000065   
4   112.839667    208.860000    107.392833        0.004850          0.000042   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0  0.004998  0.007312    0.014992      0.053932          0.515167  ...   
1  0.003508  0.005035    0.010527      0.029677          0.288667  ...   
2  0.001463  0.001425    0.004393      0.017093          0.151333  ...   
3  0.005090  0.004368    0.015263      0.042640          0.407333  ...   
4  0.002468  0.002817    0.007405      0.025015          0.224500  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delt

In [ ]:
import pandas as pd

meg_dataset = pd.read_csv('/content/drive/MyDrive/final_meg_rest_21_par-lev_final.csv')

print("Loaded data shape:", meg_dataset.shape)
print(meg_dataset.head())

Loaded data shape: (127, 106)
   burst_rate_L_mean  burst_rate_L_std  burst_rate_R_mean  burst_rate_R_std  \
0           4.412365          0.242544           4.486517          0.188241   
1           3.828010          0.316999           3.783657          0.297773   
2           4.645359          0.130894           4.713304          0.111269   
3           4.142208          0.243497           4.183916          0.217601   
4           3.923527          0.336133           4.035406          0.306960   

   burst_rate_asym  mean_burst_duration_L_mean  mean_burst_duration_L_std  \
0        -0.008333                    0.056832                   0.003237   
1         0.005827                    0.065725                   0.005382   
2        -0.007260                    0.053844                   0.001551   
3        -0.005009                    0.060555                   0.003554   
4        -0.014057                    0.064178                   0.005454   

   mean_burst_duration_R_mean  m

Experiment	Residual block	DANN / GRL	SCL	Purpose
A2: Residual encoder + classifier

In [ ]:
# ============================================================
# MIRA-NET COMPONENT ABLATION A2
# RESIDUAL ENCODER + CLASSIFIER
#
# Residual block: ON
# DANN / GRL:     OFF
# SCL:            OFF
#
# Retained from MIRA-Net:
#   1. Stratified 5-fold cross-validation
#   2. Training-fold-only numerical preprocessing
#   3. RF-SHAP Top-7 feature selection per modality
#   4. Separate training-fold StandardScaler per modality
#   5. Shared 32-D residual encoder
#   6. Classification cross-entropy training
#   7. Final MLP classifier
#   8. OOF predictions and 32-D latent embeddings
#   9. Fold-wise metrics and CSV exports
#
# Assumptions:
#   acoustic_dataset and meg_dataset are pandas DataFrames
#   Last column contains labels: 0 = HC, 1 = PD
#   Participant_ID columns have already been removed
# ============================================================

import random
import warnings

import numpy as np
import pandas as pd
import shap

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# DEVICE AND REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)


# ============================================================
# MODEL SETTINGS
# ============================================================

LATENT_DIM = 32
TOP_K = 7
N_SPLITS = 5

RF_N_ESTIMATORS = 300
NUMERICAL_CAP = 1e12

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Keep these settings identical across ablation experiments.
ENCODER_EPOCHS = 5
MLP_EPOCHS = 5
MLP_LEARNING_RATE = 1e-3


# ============================================================
# LOAD FULL-FEATURE DATASETS
# ============================================================

df_acoustic_full = acoustic_dataset.copy()
df_meg_full = meg_dataset.copy()


def extract_features_and_labels(df, dataset_name):

    if df.shape[1] < 2:
        raise ValueError(
            f"{dataset_name} must contain features and a label."
        )

    y = pd.to_numeric(
        df.iloc[:, -1],
        errors="raise"
    ).to_numpy(dtype=np.int64)

    X_df = df.iloc[:, :-1].apply(
        pd.to_numeric,
        errors="coerce"
    )

    X = X_df.to_numpy(dtype=np.float64)

    X[~np.isfinite(X)] = np.nan

    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            f"{dataset_name}: labels must be 0 and 1."
        )

    if X.shape[1] == 0:
        raise ValueError(
            f"{dataset_name}: no features found."
        )

    class_counts = np.bincount(y, minlength=2)

    if len(y) < N_SPLITS or class_counts.min() < N_SPLITS:
        raise ValueError(
            f"{dataset_name}: each class must contain at least "
            f"{N_SPLITS} samples for stratified CV."
        )

    return X, y, list(X_df.columns)


(
    X_acoustic_raw,
    y_acoustic,
    acoustic_feature_names
) = extract_features_and_labels(
    df_acoustic_full,
    "Acoustic"
)

(
    X_meg_raw,
    y_meg,
    meg_feature_names
) = extract_features_and_labels(
    df_meg_full,
    "MEG"
)

print("\nDataset shapes")
print("Acoustic:", X_acoustic_raw.shape)
print("MEG     :", X_meg_raw.shape)

print(
    "Acoustic labels:",
    dict(zip(*np.unique(y_acoustic, return_counts=True)))
)

print(
    "MEG labels:",
    dict(zip(*np.unique(y_meg, return_counts=True)))
)


# ============================================================
# LEAKAGE-CONTROLLED NUMERICAL PREPROCESSING
# ============================================================

def preprocess_fold_train_test(X_train, X_test):

    X_train = np.asarray(
        X_train,
        dtype=np.float64
    ).copy()

    X_test = np.asarray(
        X_test,
        dtype=np.float64
    ).copy()

    if (
        X_train.ndim != 2
        or X_test.ndim != 2
        or X_train.shape[1] != X_test.shape[1]
    ):
        raise ValueError(
            "Train/test features must be 2-D with matching "
            "feature counts."
        )

    # Replace non-finite values with NaN.
    X_train[~np.isfinite(X_train)] = np.nan
    X_test[~np.isfinite(X_test)] = np.nan

    # Apply fixed numerical safety cap.
    X_train = np.clip(
        X_train,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    X_test = np.clip(
        X_test,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    n_features = X_train.shape[1]

    lower_bounds = np.zeros(
        n_features,
        dtype=np.float64
    )

    upper_bounds = np.zeros(
        n_features,
        dtype=np.float64
    )

    medians = np.zeros(
        n_features,
        dtype=np.float64
    )

    # Learn clipping bounds from training data only.
    for j in range(n_features):

        valid_values = X_train[:, j][
            np.isfinite(X_train[:, j])
        ]

        if valid_values.size == 0:
            lower_bounds[j] = 0.0
            upper_bounds[j] = 0.0
        else:
            lower_bounds[j] = np.quantile(
                valid_values,
                0.01
            )

            upper_bounds[j] = np.quantile(
                valid_values,
                0.99
            )

    # Apply training-derived bounds to train and test.
    X_train = np.maximum(
        np.minimum(X_train, upper_bounds),
        lower_bounds
    )

    X_test = np.maximum(
        np.minimum(X_test, upper_bounds),
        lower_bounds
    )

    # Learn medians from training data only.
    for j in range(n_features):

        valid_values = X_train[
            np.isfinite(X_train[:, j]),
            j
        ]

        medians[j] = (
            np.median(valid_values)
            if valid_values.size
            else 0.0
        )

    def impute_with_training_medians(X):

        X = np.asarray(
            X,
            dtype=np.float64
        ).copy()

        X[~np.isfinite(X)] = np.nan

        for j in range(X.shape[1]):

            missing = ~np.isfinite(X[:, j])

            X[missing, j] = medians[j]

        return X

    X_train_clean = impute_with_training_medians(
        X_train
    )

    X_test_clean = impute_with_training_medians(
        X_test
    )

    if (
        not np.isfinite(X_train_clean).all()
        or not np.isfinite(X_test_clean).all()
    ):
        raise ValueError(
            "Non-finite values remain after preprocessing."
        )

    return (
        X_train_clean,
        X_test_clean,
        medians,
        lower_bounds,
        upper_bounds
    )


# ============================================================
# FOLD-WISE RF-SHAP TOP-7 FEATURE SELECTION
# ============================================================

def select_top_features_rf_shap(
    X_train,
    y_train,
    feature_names,
    top_k=7,
    random_state=42,
    dataset_name="Dataset"
):

    X_train = np.asarray(
        X_train,
        dtype=np.float64
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64
    )

    if not np.isfinite(X_train).all():
        raise ValueError(
            f"{dataset_name}: non-finite values before RF fitting."
        )

    k = min(top_k, X_train.shape[1])

    if k < 1:
        raise ValueError(
            f"{dataset_name}: no features available."
        )

    rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        random_state=random_state,
        class_weight="balanced",
        n_jobs=-1
    )

    # Fit RF on outer-training participants only.
    rf.fit(X_train, y_train)

    # Calculate SHAP values on training observations only.
    explainer = shap.TreeExplainer(rf)

    shap_values = explainer.shap_values(
        X_train
    )

    # Handle different SHAP output formats.
    if isinstance(shap_values, list):

        class_idx = (
            1 if len(shap_values) > 1 else 0
        )

        sv = np.asarray(
            shap_values[class_idx]
        )

    else:

        sv = np.asarray(shap_values)

        if sv.ndim == 3:

            # Shape: samples x features x classes.
            if (
                sv.shape[0] == len(X_train)
                and sv.shape[1] == X_train.shape[1]
            ):

                class_idx = min(
                    1,
                    sv.shape[2] - 1
                )

                sv = sv[:, :, class_idx]

            # Shape: classes x samples x features.
            elif (
                sv.shape[1] == len(X_train)
                and sv.shape[2] == X_train.shape[1]
            ):

                class_idx = min(
                    1,
                    sv.shape[0] - 1
                )

                sv = sv[class_idx, :, :]

            else:
                raise ValueError(
                    f"{dataset_name}: unexpected SHAP shape "
                    f"{sv.shape}."
                )

    if sv.shape != X_train.shape:
        raise ValueError(
            f"{dataset_name}: SHAP shape {sv.shape} does not "
            f"match training data shape {X_train.shape}."
        )

    if not np.isfinite(sv).all():
        raise ValueError(
            f"{dataset_name}: SHAP returned non-finite values."
        )

    # Mean absolute SHAP importance from training fold.
    importance = np.mean(
        np.abs(sv),
        axis=0
    )

    selected_indices = np.argsort(
        importance
    )[::-1][:k]

    selected_names = [
        feature_names[i]
        for i in selected_indices
    ]

    print(
        f"\n{dataset_name} RF-SHAP Top-{k} features:"
    )

    for rank, idx in enumerate(
        selected_indices,
        start=1
    ):

        print(
            f"{rank:02d}. {feature_names[idx]} "
            f"(mean |SHAP| = {importance[idx]:.6f})"
        )

    return selected_indices, selected_names


# ============================================================
# RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )


# ============================================================
# A2: RESIDUAL ENCODER + CLASSIFIER
#
# Residual block: ON
# DANN / GRL: OFF
# SCL: OFF
#
# The encoder is trained using classification cross-entropy only.
# ============================================================

class ResidualEncoderClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=32
    ):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        self.task_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        z = self.encoder(x)

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        task_logits = self.task_head(
            z_norm
        )

        return task_logits, z_norm


# ============================================================
# TRAIN RESIDUAL ENCODER ON OUTER-TRAINING PARTICIPANTS
# ============================================================

def train_residual_encoder(
    X_acoustic_train,
    y_acoustic_train,
    X_meg_train,
    y_meg_train,
    fold_number
):

    # Pool training participants from both modalities.
    X_train_all = np.vstack([
        X_acoustic_train,
        X_meg_train
    ]).astype(np.float32)

    y_train_all = np.hstack([
        y_acoustic_train,
        y_meg_train
    ]).astype(np.int64)

    if not np.isfinite(X_train_all).all():
        raise ValueError(
            f"Non-finite encoder input in fold {fold_number}."
        )

    dataset = TensorDataset(
        torch.tensor(
            X_train_all,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train_all,
            dtype=torch.long
        )
    )

    loader_generator = torch.Generator()

    loader_generator.manual_seed(
        SEED + fold_number
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=loader_generator
    )

    model = ResidualEncoderClassifier(
        input_dim=X_train_all.shape[1],
        latent_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    task_loss_fn = nn.CrossEntropyLoss()

    print(
        f"\nTraining A2 Residual Encoder | "
        f"Outer fold {fold_number}"
    )

    for epoch in range(ENCODER_EPOCHS):

        model.train()

        running_loss = 0.0
        total_correct = 0
        total_samples = 0

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            task_logits, _ = model(xb)

            # Classification loss only.
            loss = task_loss_fn(
                task_logits,
                yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite encoder loss in fold "
                    f"{fold_number}, epoch {epoch + 1}."
                )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predictions = task_logits.argmax(
                dim=1
            )

            total_correct += (
                predictions == yb
            ).sum().item()

            total_samples += len(yb)

        epoch_loss = (
            running_loss / max(len(loader), 1)
        )

        train_accuracy = (
            total_correct / max(total_samples, 1)
        )

        print(
            f"Epoch {epoch + 1:02d}/{ENCODER_EPOCHS} "
            f"| Loss={epoch_loss:.4f} "
            f"| TrainAcc={train_accuracy:.4f}"
        )

    return model


# ============================================================
# EXTRACT 32-D LATENT FEATURES
# ============================================================

def extract_latent(model, X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    if not np.isfinite(X).all():
        raise ValueError(
            "Non-finite values passed to latent extraction."
        )

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        _, z = model(X_tensor)

    z = z.cpu().numpy().astype(
        np.float32
    )

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite latent embeddings generated."
        )

    return z


# ============================================================
# FINAL MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)


# ============================================================
# TRAIN FINAL MLP AND EVALUATE HELD-OUT PARTICIPANTS
# ============================================================

def train_and_evaluate_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    fold_number
):

    X_train = np.asarray(
        X_train,
        dtype=np.float32
    )

    X_test = np.asarray(
        X_test,
        dtype=np.float32
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64
    )

    y_test = np.asarray(
        y_test,
        dtype=np.int64
    )

    if (
        not np.isfinite(X_train).all()
        or not np.isfinite(X_test).all()
    ):
        raise ValueError(
            "Non-finite latent values passed to MLP."
        )

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
        device=device
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
        device=device
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long,
        device=device
    )

    classifier = FinalMLP(
        input_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        classifier.parameters(),
        lr=MLP_LEARNING_RATE
    )

    criterion = nn.CrossEntropyLoss()

    # Train the MLP on training-fold latent embeddings only.
    classifier.train()

    for epoch in range(MLP_EPOCHS):

        optimizer.zero_grad()

        logits = classifier(
            X_train_tensor
        )

        loss = criterion(
            logits,
            y_train_tensor
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite MLP loss in fold {fold_number}."
            )

        loss.backward()
        optimizer.step()

    # Evaluate on held-out participants only.
    classifier.eval()

    with torch.no_grad():

        logits = classifier(
            X_test_tensor
        )

        probabilities = F.softmax(
            logits,
            dim=1
        )[:, 1].cpu().numpy()

        predictions = logits.argmax(
            dim=1
        ).cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),

        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "AUC": (
            roc_auc_score(y_test, probabilities)
            if len(np.unique(y_test)) == 2
            else np.nan
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,

        "predictions": predictions,
        "probabilities": probabilities
    }

    print(
        f"\nA2 Fold {fold_number} held-out metrics:"
    )

    for metric in [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "AUC"
    ]:

        print(
            f"{metric:12s}: {metrics[metric]:.4f}"
        )

    print(
        "Confusion matrix [[TN, FP], [FN, TP]]: "
        f"[[{tn}, {fp}], [{fn}, {tp}]]"
    )

    return metrics


# ============================================================
# OUTER STRATIFIED CROSS-VALIDATION SPLITS
# ============================================================

acoustic_splits = list(
    StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(
        X_acoustic_raw,
        y_acoustic
    )
)

meg_splits = list(
    StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(
        X_meg_raw,
        y_meg
    )
)


# ============================================================
# INITIALIZE OOF ARRAYS
# ============================================================

oof_pred_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_prob_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.float32
)

oof_pred_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

oof_prob_meg = np.zeros(
    len(y_meg),
    dtype=np.float32
)

oof_count_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_count_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

fused_acoustic_32D = np.zeros(
    (len(y_acoustic), LATENT_DIM),
    dtype=np.float32
)

fused_meg_32D = np.zeros(
    (len(y_meg), LATENT_DIM),
    dtype=np.float32
)

fold_records = []

selected_features_by_fold = {}


# ============================================================
# OUTER FOLD LOOP
# ============================================================

print("\n" + "=" * 75)
print(
    "A2: RESIDUAL ENCODER + CLASSIFIER "
    "| Residual ON | DANN/GRL OFF | SCL OFF"
)
print("=" * 75)

for fold in range(N_SPLITS):

    fold_number = fold + 1

    print("\n" + "=" * 75)
    print(
        f"A2 OUTER FOLD {fold_number}/{N_SPLITS}"
    )
    print("=" * 75)

    # --------------------------------------------------------
    # 1. Outer train/test splits
    # --------------------------------------------------------

    train_a_idx, test_a_idx = acoustic_splits[fold]
    train_m_idx, test_m_idx = meg_splits[fold]

    X_a_train_raw = X_acoustic_raw[train_a_idx]
    X_a_test_raw = X_acoustic_raw[test_a_idx]

    y_a_train = y_acoustic[train_a_idx]
    y_a_test = y_acoustic[test_a_idx]

    X_m_train_raw = X_meg_raw[train_m_idx]
    X_m_test_raw = X_meg_raw[test_m_idx]

    y_m_train = y_meg[train_m_idx]
    y_m_test = y_meg[test_m_idx]

    print(
        f"Acoustic train/test: "
        f"{len(train_a_idx)}/{len(test_a_idx)}"
    )

    print(
        f"MEG train/test     : "
        f"{len(train_m_idx)}/{len(test_m_idx)}"
    )

    # --------------------------------------------------------
    # 2. Train-only preprocessing
    # --------------------------------------------------------

    (
        X_a_train_clean,
        X_a_test_clean,
        _,
        _,
        _
    ) = preprocess_fold_train_test(
        X_a_train_raw,
        X_a_test_raw
    )

    (
        X_m_train_clean,
        X_m_test_clean,
        _,
        _,
        _
    ) = preprocess_fold_train_test(
        X_m_train_raw,
        X_m_test_raw
    )

    # --------------------------------------------------------
    # 3. Acoustic RF-SHAP Top-7
    # --------------------------------------------------------

    a_selected_idx, a_selected_names = (
        select_top_features_rf_shap(
            X_train=X_a_train_clean,
            y_train=y_a_train,
            feature_names=acoustic_feature_names,
            top_k=TOP_K,
            random_state=SEED + fold_number,
            dataset_name=f"Acoustic fold {fold_number}"
        )
    )

    # --------------------------------------------------------
    # 4. MEG RF-SHAP Top-7
    # --------------------------------------------------------

    m_selected_idx, m_selected_names = (
        select_top_features_rf_shap(
            X_train=X_m_train_clean,
            y_train=y_m_train,
            feature_names=meg_feature_names,
            top_k=TOP_K,
            random_state=SEED + 100 + fold_number,
            dataset_name=f"MEG fold {fold_number}"
        )
    )

    if len(a_selected_idx) != len(m_selected_idx):
        raise ValueError(
            "Acoustic and MEG must have equal selected feature "
            "counts for the shared encoder."
        )

    selected_features_by_fold[fold_number] = {
        "acoustic": a_selected_names,
        "meg": m_selected_names
    }

    # --------------------------------------------------------
    # 5. Fold-wise standardization
    # --------------------------------------------------------

    scaler_acoustic = StandardScaler()

    X_a_train = scaler_acoustic.fit_transform(
        X_a_train_clean[:, a_selected_idx]
    ).astype(np.float32)

    X_a_test = scaler_acoustic.transform(
        X_a_test_clean[:, a_selected_idx]
    ).astype(np.float32)

    scaler_meg = StandardScaler()

    X_m_train = scaler_meg.fit_transform(
        X_m_train_clean[:, m_selected_idx]
    ).astype(np.float32)

    X_m_test = scaler_meg.transform(
        X_m_test_clean[:, m_selected_idx]
    ).astype(np.float32)

    for name, arr in [
        ("Acoustic train", X_a_train),
        ("Acoustic test", X_a_test),
        ("MEG train", X_m_train),
        ("MEG test", X_m_test)
    ]:

        if not np.isfinite(arr).all():
            raise RuntimeError(
                f"{name} contains non-finite values "
                f"in fold {fold_number}."
            )

    # --------------------------------------------------------
    # 6. Train residual encoder
    #
    # Classification loss only.
    # No DANN, GRL, or SCL.
    # --------------------------------------------------------

    model = train_residual_encoder(
        X_acoustic_train=X_a_train,
        y_acoustic_train=y_a_train,
        X_meg_train=X_m_train,
        y_meg_train=y_m_train,
        fold_number=fold_number
    )

    # --------------------------------------------------------
    # 7. Extract 32-D latent embeddings
    # --------------------------------------------------------

    z_a_train = extract_latent(
        model,
        X_a_train
    )

    z_m_train = extract_latent(
        model,
        X_m_train
    )

    z_a_test = extract_latent(
        model,
        X_a_test
    )

    z_m_test = extract_latent(
        model,
        X_m_test
    )

    X_latent_train = np.vstack([
        z_a_train,
        z_m_train
    ]).astype(np.float32)

    y_latent_train = np.hstack([
        y_a_train,
        y_m_train
    ]).astype(np.int64)

    X_latent_test = np.vstack([
        z_a_test,
        z_m_test
    ]).astype(np.float32)

    y_latent_test = np.hstack([
        y_a_test,
        y_m_test
    ]).astype(np.int64)

    # --------------------------------------------------------
    # 8. Train final MLP and evaluate held-out participants
    # --------------------------------------------------------

    result = train_and_evaluate_mlp(
        X_train=X_latent_train,
        y_train=y_latent_train,
        X_test=X_latent_test,
        y_test=y_latent_test,
        fold_number=fold_number
    )

    fold_records.append({
        "Fold": fold_number,
        **{
            key: result[key]
            for key in [
                "Accuracy",
                "Precision",
                "Recall",
                "Specificity",
                "F1",
                "AUC",
                "TN",
                "FP",
                "FN",
                "TP"
            ]
        }
    })

    # --------------------------------------------------------
    # 9. Store held-out OOF predictions and embeddings
    # --------------------------------------------------------

    n_a_test = len(test_a_idx)

    pred_a_test = result["predictions"][:n_a_test]
    prob_a_test = result["probabilities"][:n_a_test]

    pred_m_test = result["predictions"][n_a_test:]
    prob_m_test = result["probabilities"][n_a_test:]

    oof_pred_acoustic[test_a_idx] = pred_a_test
    oof_prob_acoustic[test_a_idx] = prob_a_test

    oof_pred_meg[test_m_idx] = pred_m_test
    oof_prob_meg[test_m_idx] = prob_m_test

    fused_acoustic_32D[test_a_idx] = z_a_test
    fused_meg_32D[test_m_idx] = z_m_test

    oof_count_acoustic[test_a_idx] += 1
    oof_count_meg[test_m_idx] += 1

    print("\nHeld-out embedding shapes")
    print("Acoustic:", z_a_test.shape)
    print("MEG     :", z_m_test.shape)


# ============================================================
# CHECK OOF ASSIGNMENTS
# ============================================================

print("\n" + "=" * 75)
print("OOF ASSIGNMENT CHECK")
print("=" * 75)

print(
    "Acoustic OOF counts:",
    np.unique(
        oof_count_acoustic,
        return_counts=True
    )
)

print(
    "MEG OOF counts:",
    np.unique(
        oof_count_meg,
        return_counts=True
    )
)

assert np.all(oof_count_acoustic == 1)
assert np.all(oof_count_meg == 1)


# ============================================================
# COMBINE OOF EMBEDDINGS AND PREDICTIONS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D,
    fused_meg_32D
]).astype(np.float32)

y_fused = np.hstack([
    y_acoustic,
    y_meg
]).astype(np.int64)

fused_oof_probs = np.hstack([
    oof_prob_acoustic,
    oof_prob_meg
]).astype(np.float32)

fused_oof_preds = np.hstack([
    oof_pred_acoustic,
    oof_pred_meg
]).astype(np.int64)


# ============================================================
# COMBINED OOF METRICS
# ============================================================

print("\n" + "=" * 75)
print("A2 COMBINED OOF RESULTS")
print("=" * 75)

print(
    f"Accuracy  : "
    f"{accuracy_score(y_fused, fused_oof_preds):.4f}"
)

print(
    f"Precision : "
    f"{precision_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"F1-score  : "
    f"{f1_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"AUC       : "
    f"{roc_auc_score(y_fused, fused_oof_probs):.4f}"
)


# ============================================================
# MEAN AND STANDARD DEVIATION ACROSS OUTER FOLDS
# ============================================================

fold_metrics_df = pd.DataFrame(
    fold_records
)

print("\nA2 per-fold metrics:")
print(
    fold_metrics_df.round(4).to_string(index=False)
)

summary_rows = []

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC"
]:

    values = fold_metrics_df[
        metric
    ].to_numpy(dtype=float)

    summary_rows.append({
        "Metric": metric,
        "Mean": np.nanmean(values),
        "SD": np.nanstd(values, ddof=1)
    })

summary_df = pd.DataFrame(
    summary_rows
)

print("\nA2 mean and SD across folds:")
print(
    summary_df.round(4).to_string(index=False)
)


# ============================================================
# FINAL LATENT SHAPES
# ============================================================

print("\n" + "=" * 75)
print("A2 FINAL OOF LATENT SHAPES")
print("=" * 75)

print(
    "Acoustic 32-D:",
    fused_acoustic_32D.shape
)

print(
    "MEG 32-D     :",
    fused_meg_32D.shape
)

print(
    "Combined 32-D:",
    X_fused.shape
)

assert fused_acoustic_32D.shape == (
    len(y_acoustic),
    LATENT_DIM
)

assert fused_meg_32D.shape == (
    len(y_meg),
    LATENT_DIM
)

assert X_fused.shape == (
    len(y_acoustic) + len(y_meg),
    LATENT_DIM
)

assert np.isfinite(X_fused).all()


# ============================================================
# SAVE OOF EMBEDDINGS AND PREDICTIONS
# ============================================================

latent_columns = [
    f"latent_{i+1}"
    for i in range(LATENT_DIM)
]

oof_acoustic_df = pd.DataFrame(
    fused_acoustic_32D,
    columns=latent_columns
)

oof_acoustic_df["Status"] = y_acoustic
oof_acoustic_df["OOF_Prediction"] = oof_pred_acoustic
oof_acoustic_df["OOF_Probability"] = oof_prob_acoustic

oof_meg_df = pd.DataFrame(
    fused_meg_32D,
    columns=latent_columns
)

oof_meg_df["Status"] = y_meg
oof_meg_df["OOF_Prediction"] = oof_pred_meg
oof_meg_df["OOF_Probability"] = oof_prob_meg

oof_acoustic_df.to_csv(
    "mira_net_A2_acoustic_oof_embeddings.csv",
    index=False
)

oof_meg_df.to_csv(
    "mira_net_A2_meg_oof_embeddings.csv",
    index=False
)

fold_metrics_df.to_csv(
    "mira_net_A2_fold_metrics.csv",
    index=False
)

summary_df.to_csv(
    "mira_net_A2_fold_summary.csv",
    index=False
)


# ============================================================
# SAVE FOLD-SPECIFIC RF-SHAP FEATURES
# ============================================================

feature_rows = []

for fold_number, modalities in selected_features_by_fold.items():

    for modality in ["acoustic", "meg"]:

        for rank, feature_name in enumerate(
            modalities[modality],
            start=1
        ):

            feature_rows.append({
                "Fold": fold_number,
                "Modality": modality,
                "Rank": rank,
                "Feature": feature_name
            })

selected_features_df = pd.DataFrame(
    feature_rows
)

selected_features_df.to_csv(
    "mira_net_A2_rf_shap_selected_features.csv",
    index=False
)


# ============================================================
# COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("A2 ABLATION COMPLETE")
print("=" * 75)

print("Residual block: ON")
print("DANN / GRL    : OFF")
print("SCL           : OFF")

print(
    "Encoder epochs:",
    ENCODER_EPOCHS,
    "| Final MLP epochs:",
    MLP_EPOCHS
)

print(
    "Acoustic 32-D:",
    fused_acoustic_32D.shape
)

print(
    "MEG 32-D     :",
    fused_meg_32D.shape
)

print(
    "Combined 32-D:",
    X_fused.shape
)

print("\nSaved files:")
print(" - mira_net_A2_acoustic_oof_embeddings.csv")
print(" - mira_net_A2_meg_oof_embeddings.csv")
print(" - mira_net_A2_fold_metrics.csv")
print(" - mira_net_A2_fold_summary.csv")
print(" - mira_net_A2_rf_shap_selected_features.csv")

Device: cpu

Dataset shapes
Acoustic: (193, 69)
MEG     : (127, 105)
Acoustic labels: {np.int64(0): np.int64(66), np.int64(1): np.int64(127)}
MEG labels: {np.int64(0): np.int64(65), np.int64(1): np.int64(62)}

A2: RESIDUAL ENCODER + CLASSIFIER | Residual ON | DANN/GRL OFF | SCL OFF

A2 OUTER FOLD 1/5
Acoustic train/test: 154/39
MEG train/test     : 101/26

Acoustic fold 1 RF-SHAP Top-7 features:
01. Degree-voice breaks (mean |SHAP| = 0.020099)
02. MDVP:Shimmer (mean |SHAP| = 0.019161)
03. Delta1 (mean |SHAP| = 0.019100)
04. RPDE (mean |SHAP| = 0.018424)
05. Delta0 (mean |SHAP| = 0.017792)
06. No. of voice breaks (mean |SHAP| = 0.017317)
07. MFCC5 (mean |SHAP| = 0.017233)

MEG fold 1 RF-SHAP Top-7 features:
01. burst_rate_asym (mean |SHAP| = 0.019693)
02. burst_dur_cv_R_std (mean |SHAP| = 0.014527)
03. burst_dur_cv_L_std (mean |SHAP| = 0.013783)
04. ibi_cv_asym (mean |SHAP| = 0.013600)
05. mean_burst_duration_R_std (mean |SHAP| = 0.013082)
06. mean_ibi_R_std (mean |SHAP| = 0.012419)
07.

In [ ]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()
fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor, fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([320, 32])
torch.float32


In [ ]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = acoustic_dataset.iloc[:, -1].values.astype(np.int32)   # (7350,)

# MEG target from final_meg_df (last column)
y_meg = meg_dataset.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)
meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([acoustic_full, meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([320, 32]), dtype: torch.float32
Labels shape: torch.Size([320]), dtype: torch.int64


In [ ]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(320, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  0.000852  0.160750  0.005471  0.117661  0.027260 -0.077167 -0.355221   
1  0.253017  0.087402 -0.066207 -0.075083 -0.330305 -0.086777 -0.075101   
2  0.050101  0.198926  0.024044  0.174544  0.010179 -0.073376 -0.288008   
3  0.102710 -0.270563  0.291122  0.110311  0.251135  0.209994 -0.291556   
4  0.111572 -0.303669  0.141032  0.195585 -0.037902 -0.047305 -0.150884   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0 -0.102311  0.272302  0.235828  ... -0.288266  0.188228 -0.258178  0.121008   
1 -0.256585  0.018530  0.076754  ... -0.101803 -0.054820  0.040284 -0.323179   
2 -0.052570  0.236252  0.256960  ... -0.209810  0.239172 -0.268980  0.057409   
3  0.049344  0.099628  0.143152  ...  0.129085 -0.055715  0.399256  0.082575   
4  0.030797  0.001545 -0.184765  ...  0.072848 -0.275601  0.156944  0.248421   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [ ]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:

# ============================================================
# IMPROVED ENSEMBLE CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. XGBoost
#   2. Random Forest
#   3. HistGradientBoosting
#   4. Weighted Soft Voting
#   5. Stacking
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
# ============================================================

!pip install -q xgboost scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    scale_pos_weight = n_negative / max(n_positive, 1)

    xgb = XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=2500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    hgb = HistGradientBoostingClassifier(
        max_iter=2500,
        learning_rate=0.04,
        max_leaf_nodes=15,
        max_depth=5,
        min_samples_leaf=10,
        l2_regularization=2.0,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    )

    voting = VotingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        voting="soft",
        weights=[2, 1, 1],
        n_jobs=1
    )

    stacking = StackingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        final_estimator=LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced"
        ),
        stack_method="predict_proba",
        passthrough=False,
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=123
        ),
        n_jobs=1
    )

    return {
        "XGBoost": xgb,
        "Random Forest": rf,
        "HistGradientBoosting": hgb,
        "Voting Fusion": voting,
        "Stacking Fusion": stacking
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Bootstrap CIs for F1 and AUC ----------------
    # Stratified resampling preserves both class counts.
    # Predictions and probabilities remain paired with labels.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Fit imputation only on this fold's training partition.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        models = create_models(y_tr)
        model = models[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = model.predict(X_val_imp)
        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        # Print all six metrics and their CIs for each fold.
        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Average lower and upper fold CI endpoints.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_models = create_models(y_train)
    final_model = final_models[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = final_model.predict(X_test_imp)
    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: VOTING ENSEMBLE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("WEIGHTED VOTING ENSEMBLE RESULTS")
print("=" * 100)

print("Voting weights: XGBoost=2, Random Forest=1, HGB=1")

print("\nVoting fold results:")
display(
    fold_results_df[
        fold_results_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting cross-validation summary:")
display(
    cv_summary_df[
        cv_summary_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting independent test results:")
display(
    test_results_df.loc[["Voting Fusion"]].round(4)
)

X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

EVALUATING: XGBoost

Fold 01/10
Accuracy       : 0.6207 (95% CI: 0.4400-0.7731)
Precision      : 0.6250 (95% CI: 0.4271-0.7884)
Sensitivity    : 0.8824 (95% CI: 0.6566-0.9671)
Specificity    : 0.2500 (95% CI: 0.0889-0.5323)
F1             : 0.7317 (95% CI: 0.6154-0.8205)
AUC            : 0.7059 (95% CI: 0.4902-0.9020)

Fold 02/10
Accuracy       : 0.4828 (95% CI: 0.3139-0.6557)
Precision      : 0.5625 (95% CI: 0.3318-0.7690)
Sensitivity    : 0.5294 (95% CI: 0.3096-0.7383)
Specificity    : 0.4167 (95% CI: 0.1933-0.6805)
F1             : 0.5455 (95% CI: 0.3448-0.7222)
AUC            : 0.5539 (95% CI: 0.3382-0.7647)

Fold 03/10
Accuracy       : 0.5862 (95% CI: 0.4074-0.7449)
Precision      : 0.7273 (95% CI: 0.4344-0.9025)
Sensitivity    : 0.4706 (95% CI: 0.2617-0.6904)
Specificity    : 0.7500 (95% CI: 0.4677-0.9111)
F1             : 0.5714 (

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
20,HistGradientBoosting,1,0.6897,0.5077,0.8272,0.6818,0.4732,0.8364,0.8824,0.6566,...,0.8718,0.7059,0.4754,0.8971,0.6495,0.3442,5,7,2,15
21,HistGradientBoosting,2,0.5862,0.4074,0.7449,0.6471,0.4130,0.8269,0.6471,0.4130,...,0.8000,0.6716,0.4558,0.8529,0.5735,0.1471,6,6,6,11
22,HistGradientBoosting,3,0.6207,0.4400,0.7731,0.8000,0.4902,0.9433,0.4706,0.2617,...,0.7857,0.7549,0.5686,0.9069,0.6520,0.3149,10,2,9,8
23,HistGradientBoosting,4,0.6552,0.4735,0.8006,0.6842,0.4601,0.8464,0.7647,0.5274,...,0.8571,0.6471,0.4216,0.8529,0.6324,0.2743,6,6,4,13
24,HistGradientBoosting,5,0.7241,0.5428,0.8530,0.7143,0.5004,0.8619,0.8824,0.6566,...,0.8947,0.7206,0.5195,0.8874,0.6912,0.4213,6,6,2,15
25,HistGradientBoosting,6,0.5517,0.3755,0.7159,0.6000,0.3866,0.7812,0.7059,0.4687,...,0.7805,0.5833,0.3382,0.7844,0.5196,0.0417,4,8,5,12
26,HistGradientBoosting,7,0.6207,0.4400,0.7731,0.6875,0.4440,0.8584,0.6471,0.4130,...,0.8235,0.6520,0.4216,0.8480,0.6152,0.2282,7,5,6,11
27,HistGradientBoosting,8,0.8621,0.6944,0.9450,0.8421,0.6243,0.9448,0.9412,0.7302,...,0.9714,0.8186,0.6324,0.9608,0.8456,0.7162,9,3,1,16
28,HistGradientBoosting,9,0.7857,0.6046,0.8979,0.7895,0.5667,0.9149,0.8824,0.6566,...,0.9412,0.7968,0.6150,0.9465,0.7594,0.5424,7,4,2,15
29,HistGradientBoosting,10,0.7857,0.6046,0.8979,0.8235,0.5897,0.9381,0.8235,0.5897,...,0.9412,0.8182,0.6310,0.9572,0.7754,0.5508,8,3,3,14



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,XGBoost,0.6256,0.0806,0.4461,0.7754,0.6848,0.0854,0.4506,0.8480,0.7000,...,0.5116,0.8247,0.6828,0.0831,0.4729,0.8652,0.6098,0.0875,0.2260,0.1709
1,Random Forest,0.6463,0.0701,0.4656,0.7924,0.6936,0.0676,0.4625,0.8541,0.7294,...,0.5392,0.8410,0.7100,0.0777,0.5041,0.8825,0.6287,0.0737,0.2656,0.1448
2,HistGradientBoosting,0.6882,0.0999,0.5091,0.8229,0.7270,0.0816,0.4948,0.8752,0.7647,...,0.5847,0.8667,0.7169,0.0800,0.5079,0.8894,0.6714,0.0986,0.3581,0.2038
3,Voting Fusion,0.6605,0.0850,0.4803,0.8027,0.7072,0.0763,0.4738,0.8642,0.7294,...,0.5501,0.8461,0.7146,0.0788,0.5091,0.8855,0.6458,0.0841,0.3008,0.1701
4,Stacking Fusion,0.6709,0.0776,0.4903,0.8112,0.7465,0.0960,0.4999,0.8900,0.6941,...,0.5277,0.8523,0.7212,0.0797,0.5168,0.8876,0.6664,0.0838,0.3397,0.1621



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Voting Fusion,1,0.6897,0.5077,0.8272,0.6818,0.4732,0.8364,0.8824,0.6566,...,0.8649,0.7402,0.5391,0.9118,0.6495,0.3442,5,7,2,15
31,Voting Fusion,2,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7648,0.6176,0.4020,0.8187,0.5441,0.0874,6,6,7,10
32,Voting Fusion,3,0.5517,0.3755,0.7159,0.6667,0.3906,0.8619,0.4706,0.2617,...,0.7333,0.7010,0.4951,0.8676,0.5686,0.1373,8,4,9,8
33,Voting Fusion,4,0.6552,0.4735,0.8006,0.6842,0.4601,0.8464,0.7647,0.5274,...,0.8571,0.7010,0.4951,0.8775,0.6324,0.2743,6,6,4,13
34,Voting Fusion,5,0.7241,0.5428,0.8530,0.7143,0.5004,0.8619,0.8824,0.6566,...,0.8947,0.7304,0.5196,0.9020,0.6912,0.4213,6,6,2,15
35,Voting Fusion,6,0.5517,0.3755,0.7159,0.6000,0.3866,0.7812,0.7059,0.4687,...,0.7805,0.6078,0.3725,0.8088,0.5196,0.0417,4,8,5,12
36,Voting Fusion,7,0.6552,0.4735,0.8006,0.7333,0.4805,0.8910,0.6471,0.4130,...,0.8387,0.6373,0.4118,0.8333,0.6569,0.3092,8,4,6,11
37,Voting Fusion,8,0.6897,0.5077,0.8272,0.7222,0.4913,0.8750,0.7647,0.5274,...,0.8718,0.7794,0.5833,0.9265,0.6740,0.3533,7,5,4,13
38,Voting Fusion,9,0.7500,0.5664,0.8732,0.7778,0.5479,0.9100,0.8235,0.5897,...,0.9143,0.7754,0.5828,0.9305,0.7299,0.4688,7,4,3,14
39,Voting Fusion,10,0.7857,0.6046,0.8979,0.8667,0.6212,0.9626,0.7647,0.5274,...,0.9412,0.8556,0.6898,0.9786,0.7914,0.5708,9,2,4,13



Voting cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
3,Voting Fusion,0.6605,0.085,0.4803,0.8027,0.7072,0.0763,0.4738,0.8642,0.7294,...,0.5501,0.8461,0.7146,0.0788,0.5091,0.8855,0.6458,0.0841,0.3008,0.1701



Voting independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
Voting Fusion,0.7188,0.5463,0.8444,0.75,0.5313,0.8881,0.7895,0.5667,0.9149,0.6154,...,0.8947,0.749,0.5587,0.915,0.7024,0.4107,8,5,4,15
